# Sample a small Bangla dialect eval set from Bhasha-Bichitra

**What this notebook does**
- Discovers the input structure (audio folder, manifest CSV, columns)
- Extracts the dialect from each file's ID (Bhasha-Bichitra encodes dialect in the filename, e.g. `valid_sandwip (7).wav` → dialect = `sandwip`)
- Samples N clips per dialect (default 5)
- Copies audio + writes matching `.reference.txt` files
- Zips the result to `/kaggle/working/dialect_eval.zip` for download → unzip locally into `new-stt/audio/dialects/`

**Prereqs (Kaggle UI)**
1. Open this notebook from the competition's **Code tab → New Notebook** button (auto-attaches the Bhasha-Bichitra data)
2. File → Import Notebook → upload this file
3. Session options → Accelerator: **None** · Internet: **Off**
4. Run All (top toolbar). Takes ~1 min.

## 1. Discover the input structure

In [ ]:
import os
from pathlib import Path

for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count('/')
    indent = '  ' * depth
    print(f'{indent}{Path(root).name}/')
    for f in files[:5]:
        print(f'{indent}  {f}')
    if len(files) > 5:
        print(f'{indent}  ... ({len(files)-5} more)')

In [ ]:
# Auto-detect: input root, manifest CSV, audio folder + extension.
INPUT_ROOT = next(Path('/kaggle/input').iterdir())
print(f'INPUT_ROOT = {INPUT_ROOT}')

csvs = list(INPUT_ROOT.rglob('*.csv'))
manifest_csv = max(csvs, key=lambda p: p.stat().st_size) if csvs else None
print(f'Manifest: {manifest_csv.relative_to(INPUT_ROOT) if manifest_csv else None}')

audio_dirs = [p for p in INPUT_ROOT.rglob('*') if p.is_dir() and (any(p.glob('*.wav')) or any(p.glob('*.mp3')))]
AUDIO_DIR = audio_dirs[0] if audio_dirs else None
AUDIO_EXT = '.wav' if AUDIO_DIR and any(AUDIO_DIR.glob('*.wav')) else '.mp3'
print(f'AUDIO_DIR = {AUDIO_DIR.relative_to(INPUT_ROOT) if AUDIO_DIR else None}')
print(f'AUDIO_EXT = {AUDIO_EXT}')

In [ ]:
import pandas as pd

df = pd.read_csv(manifest_csv)
print(f'Rows: {len(df):,}')
print(f'Columns: {list(df.columns)}\n')
print(df.head(3).to_string())

# Also show a few raw audio filenames from disk — helps confirm the dialect-in-name pattern.
print('\nSample audio filenames:')
for p in list(AUDIO_DIR.iterdir())[:8]:
    print(f'  {p.name}')

## 2. Configure — filename tells us the dialect

Bhasha-Bichitra encodes the dialect INSIDE each audio filename, not as a CSV column. Example: `valid_sandwip (7).wav` → dialect is `sandwip`. The regex below extracts it.

If your data DOES have a dialect/district column (rare for this competition), just point `DIALECT_COL` at it and the notebook uses that instead.

In [ ]:
import re

# --- Edit if defaults are wrong (cell 4 above prints the actual column names) ---
ID_COL      = 'id'          # column holding the audio filename or stem
TEXT_COL    = 'sentence'    # column with the reference transcript
DIALECT_COL = None          # set to a column name if one exists; otherwise leave None

CLIPS_PER_DIALECT = 5
RANDOM_SEED       = 42

# Regex: pattern like `train_sylhet (12)` or `valid_sandwip (7)` → captures `sylhet` / `sandwip`.
# Case-insensitive, tolerates optional whitespace before the parenthesis.
DIALECT_RE = re.compile(r'^(?:train|valid|test)_([a-z]+)', re.IGNORECASE)

def dialect_from_id(id_value: str) -> str:
    m = DIALECT_RE.match(str(id_value))
    return m.group(1).lower() if m else 'unknown'

# Attach a dialect column so downstream sampling is uniform.
if DIALECT_COL and DIALECT_COL in df.columns:
    df['_dialect'] = df[DIALECT_COL].astype(str).str.strip().str.lower()
    print(f'Using column {DIALECT_COL!r} for dialect.')
else:
    df['_dialect'] = df[ID_COL].apply(dialect_from_id)
    print(f'Extracted dialect from column {ID_COL!r} via filename regex.')

print(f'\nDialect distribution:')
print(df['_dialect'].value_counts())

# Bail out early if regex found nothing — better to fail here than write garbage.
assert df['_dialect'].nunique() > 1 or DIALECT_COL, (
    'Only one dialect detected. Either the regex is wrong for this dataset, or you need to set DIALECT_COL. '
    f'Sample id values: {df[ID_COL].head(5).tolist()}'
)

## 3. Sample, copy, and zip

In [ ]:
import shutil

OUT_DIR = Path('/kaggle/working/dialect_eval')
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)

# Stratified: N per dialect (skip 'unknown' — those are labelling failures)
usable = df[df['_dialect'] != 'unknown']
sampled = (
    usable.groupby('_dialect', group_keys=False)
          .apply(lambda g: g.sample(n=min(CLIPS_PER_DIALECT, len(g)), random_state=RANDOM_SEED))
          .reset_index(drop=True)
)
print(f'Sampled {len(sampled)} clips across {sampled["_dialect"].nunique()} dialects\n')

def resolve_audio(id_value: str) -> Path | None:
    """Find the audio file. IDs sometimes include the extension, sometimes not."""
    stem = str(id_value).strip()
    # 1) Exact match (id already has extension).
    p = AUDIO_DIR / stem
    if p.exists():
        return p
    # 2) Add default extension.
    p = AUDIO_DIR / f'{stem}{AUDIO_EXT}'
    if p.exists():
        return p
    # 3) Fuzzy: any file with this stem in any subdir.
    hits = list(AUDIO_DIR.rglob(f'{stem}.*'))
    return hits[0] if hits else None

copied, missing = 0, 0
for _, row in sampled.iterrows():
    src = resolve_audio(row[ID_COL])
    if src is None:
        missing += 1
        continue

    dialect_dir = OUT_DIR / row['_dialect']
    dialect_dir.mkdir(exist_ok=True)

    # Sanitize filename — original may contain spaces/parens which are ugly locally.
    safe_stem = re.sub(r'[^a-zA-Z0-9_-]', '_', src.stem)
    dst = dialect_dir / f'{safe_stem}{src.suffix}'
    shutil.copy(src, dst)
    (dialect_dir / f'{safe_stem}.reference.txt').write_text(str(row[TEXT_COL]), encoding='utf-8')
    copied += 1

print(f'Copied: {copied}    Missing: {missing}')
!ls {OUT_DIR}

In [ ]:
zip_path = Path('/kaggle/working/dialect_eval.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive('/kaggle/working/dialect_eval', 'zip', OUT_DIR)
print(f'Zip: {zip_path}   ({zip_path.stat().st_size / (1024*1024):.1f} MB)')
print('Download from right sidebar → Output → dialect_eval.zip')

## 4. Bring it home

**Right sidebar → Output → `dialect_eval.zip` → Download**

Then locally:
```bash
cd /Users/arko/claude/new-stt
mkdir -p audio/dialects
unzip ~/Downloads/dialect_eval.zip -d audio/dialects
ls audio/dialects/    # one folder per dialect
```

Ping me when it lands — I'll extend `eval_data.py` + `evaluate.py` with a `--dialects-only` flag and we'll benchmark both models on the same clips.